In [0]:
%run "/Workspace/Optum-DBx/Project-Optum-DBx/Dbx-Transformations/Prod/Connectors_Prod"

In [0]:
%run "/Workspace/Optum-DBx/Project-Optum-DBx/Dbx-Transformations/Prod/Generic_Prod"

In [0]:
#Importing neccessary libraries
from pyspark.sql.functions import *

In [0]:
#Calling the function to connect to ADLS storage from Connectors
adls_connect()

In [0]:
#Listing all the files in Bronze layer
list_bronze_files()

In [0]:
#Reading Subgroup.csv file from Bronze layer
patient_df = read_bronze_file_csv("Patient_records")

In [0]:
check_missing_values(patient_df,patient_df.columns)

In [0]:
check_string_value_as_nan(patient_df)

Transformation Layer

In [0]:
#Data Cleaning -> Missing values default imputation
patient_df = patient_df.fillna({"Patient_name":"Visitor/NA"})

In [0]:
#Data Cleaning -> Masking Phone number (5 digits in between)
patient_df = patient_df.withColumn("patient_phone",concat(substring(col("patient_phone"),1,6),lit("*****"),substring(col("patient_phone"),-2,2)))

In [0]:
#Data Cleaning -> Calculating Age using patient birth date column
patient_df = patient_df.withColumn("patient_age", ((months_between(current_date(), col("patient_birth_date"))/12)).cast("integer"))

In [0]:
# Droping patient birth date column after calculating age
patient_df = patient_df.drop("patient_birth_date")

In [0]:
patient_df = patient_df.fillna({"patient_phone":"0000000000"})

Writing to Silver layer

In [0]:
#Writing transformed dataframe into Silver layer
write_to_silver(patient_df,"patient_S.csv")